In [1]:
import cv2
import numpy as np
from collections import defaultdict
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pandas as pd
import matplotlib.pyplot as plt
import os
from deepface import DeepFace
import datetime
import pickle

from insightface.app import FaceAnalysis

In [2]:
app = FaceAnalysis(name='buffalo_l')
#モデル指定。https://github.com/deepinsight/insightface/tree/master/model_zoo
app.prepare(ctx_id=0, det_size=(640, 640))
# app.prepare(ctx_id=0) は、GPU を使用するか CPU を使用するかの指定です。 
# GPU を使用する場合は ctx_id=0、CPU を使用する場合は ctx_id=-1 を指定します。
# 顔検出に使う入力画像サイズ（幅×高さのピクセル数）を指定するパラメータ

def get_insightface_median_db(base_path, target_folders):
    face_db = {}
    
    print("--- InsightFace 特徴量抽出（中央値DB）開始 ---")
    for label in target_folders:
        dir_path = os.path.join(base_path, label)
        if not os.path.exists(dir_path): continue
        
        folder_embeddings = []
        for img_name in os.listdir(dir_path):
            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue
            img_path = os.path.join(dir_path, img_name)
            
            img = cv2.imread(img_path)
            if img is None: continue
            
            faces = app.get(img)
            
            # 確実に1人のみ、かつスコアが高いものを抽出
            if len(faces) == 1 and faces[0].det_score > 0.6:
                folder_embeddings.append(faces[0].normed_embedding)
                
        if folder_embeddings:
            # 1. 軸0（画像方向）で中央値を算出
            mean_feat = np.mean(folder_embeddings, axis=0)
            # 2. コサイン類似度（np.dot）に最適化するため、L2正規化を再適用
            mean_feat_norm = mean_feat / np.linalg.norm(mean_feat)
            
            clean_label = os.path.basename(label)
            face_db[clean_label] = mean_feat_norm
            print(f"Registered: {clean_label} (Based on {len(folder_embeddings)} images)")
            
    return face_db

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\真希/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\真希/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\真希/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\真希/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\真希/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det

get_deepface_dbを実行して画像の特徴量を抽出

In [3]:
db_base_path="./"
model_name='Facenet'
target_folders = ["endou", "kaki","yumiki","umezawa","suzuki","ioki","hayashi","ikeda"]
target_folders = [f"menber_db/{name}" for name in target_folders]
face_db = get_insightface_median_db(db_base_path, target_folders=target_folders)

--- InsightFace 特徴量抽出（中央値DB）開始 ---
Registered: endou (Based on 10 images)
Registered: kaki (Based on 10 images)
Registered: yumiki (Based on 10 images)
Registered: umezawa (Based on 10 images)
Registered: suzuki (Based on 10 images)
Registered: ioki (Based on 9 images)
Registered: hayashi (Based on 8 images)
Registered: ikeda (Based on 9 images)


In [4]:
cache_dir = "./face_db_cache"
os.makedirs(cache_dir, exist_ok=True)
filename = datetime.datetime.now().strftime("face_db_%Y%m%d_%H%M.pkl")
save_face_db_path = os.path.join(cache_dir, filename)
with open(save_face_db_path, "wb") as f:
    pickle.dump(face_db, f)

print(f"face_db saved: {save_face_db_path}")

face_db saved: ./face_db_cache\face_db_20260621_2216.pkl


### 複数人ケースの3つの評価指標

| 指標 | 説明 | 例 |
|------|------|-----|
| **顔検出精度** | 正解の人数を正しく検出できたか | 正解2人 → 検出2人 = ○ |
| **セット照合精度** | フレーム内の全顔セットが完全一致するか | {A,B} == {A,B} = ○ |
| **個別ラベル精度** | 各顔の識別ラベルが正確か（人数違う場合も評価可能） | 正解[A,B] vs 予測[A,C] = 50% |


結果確認用

In [ ]:
# 次回以降は以下のように読み込みます
cache_dir = "./face_db_cache"
#filename = "face_db_20260614_2116.pkl"  # 例: 保存したファイル名を指定
save_face_db_path = os.path.join(cache_dir, filename)

with open(save_face_db_path, "rb") as f:
    face_db = pickle.load(f)
print("face_db loaded")
print(f"Loaded face_db keys: {list(face_db.keys())}")

face_db loaded
Loaded face_db keys: ['endou', 'kaki', 'yumiki', 'umezawa', 'suzuki', 'ioki', 'hayashi', 'ikeda']


In [ ]:
def get_insightface_all_embeddings_db(base_path, target_folders, min_det_score=0.6):
    """各フォルダ内の全正規化埋め込みを行列として返す: label -> np.array(shape=(n,d))"""
    face_db = {}
    print("--- InsightFace 全画像埋め込みDB作成開始 ---")
    for label in target_folders:
        dir_path = os.path.join(base_path, label)
        if not os.path.exists(dir_path):
            continue
        embeddings = []
        for img_name in os.listdir(dir_path):
            if not img_name.lower().endswith(('.png','.jpg','.jpeg')):
                continue
            img_path = os.path.join(dir_path, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            faces = app.get(img)
            for f in faces:
                if getattr(f, 'det_score', 1.0) >= min_det_score:
                    emb = f.normed_embedding
                    embeddings.append(emb)
        if embeddings:
            mat = np.vstack(embeddings)  # shape (n, d)
            norms = np.linalg.norm(mat, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            mat = mat / norms
            clean_label = os.path.basename(label)
            face_db[clean_label] = mat
            print(f"Registered: {clean_label} (num_examples={mat.shape[0]})")
    return face_db

def assign_unique_person_labels_max_matrix(faces, face_db, threshold):
    """faces: list with .normed_embedding and .det_score. face_db: label->np.array(n,d).
    比較は行列積で高速化し、各ラベル内の最大類似度を採用する。"""
    face_candidates = []
    for i, face in enumerate(faces):
        conf = face.det_score
        if conf <= 0.3:
            continue
        cur = face.normed_embedding
        scores = []
        for label, mat in face_db.items():
            if mat.shape[1] != cur.shape[0]:
                raise ValueError(f"Dimension mismatch DB:{mat.shape[1]} vs cur:{cur.shape[0]}")
            sims = mat.dot(cur)
            max_score = float(np.max(sims)) if sims.size > 0 else -1.0
            scores.append((max_score, label))
        if not scores: continue
        scores.sort(key=lambda x: x[0], reverse=True)
        face_candidates.append(scores)
    used_labels = set()
    pred_persons = []
    valid_candidates = [c for c in face_candidates if len(c) > 0]
    for scores in sorted(valid_candidates, key=lambda x: x[0][0], reverse=True):
        assigned = "Unknown"
        for score, label in scores:
            if score < threshold: break
            if label not in used_labels:
                assigned = label
                used_labels.add(label)
                break
        pred_persons.append(assigned)
    return sorted(pred_persons)

# 使い方例（コメント）
face_db_all = get_insightface_all_embeddings_db('./', target_folders)
#pred = assign_unique_person_labels_max_matrix(faces, face_db_all, threshold=0.5)

--- InsightFace 全画像埋め込みDB作成開始 ---
Registered: endou (num_examples=10)
Registered: kaki (num_examples=10)
Registered: yumiki (num_examples=10)
Registered: umezawa (num_examples=10)
Registered: suzuki (num_examples=10)
Registered: ioki (num_examples=9)
Registered: hayashi (num_examples=8)
Registered: ikeda (num_examples=9)


NameError: name 'faces' is not defined

In [6]:
def normalize_face_db_keys(face_db):
    """'menber_db/xxx' のようなキーを 'xxx' に正規化して返す."""
    if not isinstance(face_db, dict):
        return face_db
    return {os.path.basename(k): v for k, v in face_db.items()}

def assign_unique_person_labels_insightface(faces, face_db, threshold):
    """
    InsightFaceの検出結果(faces)と、中央値ベースのface_dbを比較し、
    一意のラベルを割り当てる（デバッグログ付き）。
    """
    face_candidates = []
    
    for i, face in enumerate(faces):
        conf = face.det_score
        if conf <= 0.3:
            continue
            
        current_feat_norm = face.normed_embedding
        scores = []
        
        for label, target_feat_norm in face_db.items():
            if target_feat_norm.shape[0] != current_feat_norm.shape[0]:
                raise ValueError(f"Dimension mismatch! DB: {target_feat_norm.shape[0]}, Test: {current_feat_norm.shape[0]}")
                
            score = float(np.dot(target_feat_norm, current_feat_norm))
            scores.append((score, label))
            
        if not scores:
            continue
            
        scores.sort(key=lambda x: x[0], reverse=True)
        face_candidates.append(scores)
        
        # 🔍 【定量デバッグ】各顔の類似度上位3件を強制出力
        top_str = ", ".join([f"{lbl}: {sc:.4f}" for sc, lbl in scores[:3]])
        # print(f"   -> [顔検出枠 {i}] 類似度 Top3: {top_str}")

    used_labels = set()
    pred_persons = []
    
    # 確信度順のソート処理（空のリスト対策を含めて安全に処理）
    valid_candidates = [c for c in face_candidates if len(c) > 0]
    
    for scores in sorted(valid_candidates, key=lambda x: x[0][0], reverse=True):
        assigned_label = "Unknown"
        for score, label in scores:
            if score < threshold:
                # 🔍 閾値で弾かれた理由をログに残す
                # print(f"   (閾値判定外: {label} は score {score:.4f} < {threshold})")
                break
            if label not in used_labels:
                assigned_label = label
                used_labels.add(label)
                break
        pred_persons.append(assigned_label)

    return sorted(pred_persons)



def evaluate_multi_person_detailed(test_images_dir, threshold,face_db,movie_num,model_name='Facenet'):
    ## No1-3ここの閾値を0.4→0.6に上げる
    """
    複数人テスト画像の詳細評価（3つの精度指標）
    
    1. 顔検出精度: 正解数と完全一致 → 人数が合っているか
    2. セット照合精度: セット一致度 → 正確な識別ができているか  
    3. 個別ラベル精度: ラベル一致度 → 各顔の識別スコア
    """
    face_db = normalize_face_db_keys(face_db)
    
    anno_file = os.path.join(test_images_dir, f"annotations_{movie_num}.csv")
    if not os.path.exists(anno_file):
        print(f"⚠️ {anno_file} が見つかりません")
        return None, None
    
    anno_df = pd.read_csv(anno_file)
    
    detection_correct = 0  # 人数が正しい
    set_correct = 0        # セットが完全一致
    label_correct = 0      # 個別ラベルが一致
    
    detailed_results = []
    
    print("=== 複数人テスト画像 詳細評価 ===\n")
    print(f"{'Image':<20} {'正解':<15} {'予測':<15} {'検出':<5} {'セット':<5} {'ラベル':<6}")
    print("-" * 75)
    
    for _, row in anno_df.iterrows():
        img_name = row['image_name']
        img_path = os.path.join(test_images_dir, img_name)
        
        if not os.path.exists(img_path):
            continue
        
        gt_persons = sorted([p.strip() for p in str(row['persons']).split('|') if p.strip()])
        # personsの列を'|'で分割してリスト化し、空白を除去してソート
        
        try:
            objs = DeepFace.represent(img_path=img_path, 
                                     model_name=model_name, 
                                     detector_backend='retinaface', 
                                     enforce_detection=False)
            
            # DeepFace.represent は dict のリストを返すことがあるため、
            # InsightFace のオブジェクト (属性 det_score, normed_embedding を持つ) に変換する
            from types import SimpleNamespace
            faces = []
            for o in objs:
                if isinstance(o, dict) and 'embedding' in o:
                    emb = np.asarray(o['embedding'], dtype=float)
                    norm = np.linalg.norm(emb)
                    normed = emb / norm if norm != 0 else emb
                    # det_score がなければ1.0（検出済み）を仮置き
                    det = float(o.get('confidence', o.get('det_score', 1.0)))
                    faces.append(SimpleNamespace(det_score=det, normed_embedding=normed))
                else:
                    faces.append(o)

            pred_persons = assign_unique_person_labels_insightface(faces, face_db, threshold)
            
            # 3つの評価指標
            detection_ok = (len(pred_persons) == len(gt_persons))  # 人数一致
            set_ok = (set(pred_persons) == set(gt_persons))        # セット一致
            label_ok = (pred_persons == gt_persons)                # 順序含め完全一致
            
            if detection_ok:
                detection_correct += 1
            if set_ok:
                set_correct += 1
            if label_ok:
                label_correct += 1
            
            # 表示
            det_mark = "✓" if detection_ok else "✗"
            set_mark = "✓" if set_ok else "✗"
            label_mark = "✓" if label_ok else "✗"
            
            gt_str = '|'.join(gt_persons)
            pred_str = '|'.join(pred_persons)
            
            print(f"{img_name:<20} {gt_str:<15} {pred_str:<15} {det_mark:<5} {set_mark:<5} {label_mark:<6}")
            
            detailed_results.append({
                'image': img_name,
                'gt': gt_str,
                'pred': pred_str,
                'detection': detection_ok,
                'set_match': set_ok,
                'label_match': label_ok
            })
        
        except Exception as e:
            import traceback
            print(f"{img_name:<20} Error: {e}")
            traceback.print_exc()
    
    # 集計
    total = len(anno_df)
    print("\n" + "="*50)
    print(f"📊 集計結果 (計 {total} 画像)")
    print("="*50)
    print(f"顔検出精度 (人数一致):    {detection_correct}/{total} ({100*detection_correct/total:.1f}%)")
    print(f"セット照合精度 (順不同):    {set_correct}/{total} ({100*set_correct/total:.1f}%)")
    print(f"ラベル精度 (順序含む完全一致): {label_correct}/{total} ({100*label_correct/total:.1f}%)")
    
    summary_df = pd.DataFrame([{
        'detection_count': detection_correct,
        'detection_rate': 100 * detection_correct / total if total else 0.0,
        'set_count': set_correct,
        'set_rate': 100 * set_correct / total if total else 0.0,
        'label_count': label_correct,
        'label_rate': 100 * label_correct / total if total else 0.0,
    }])
    
    return pd.DataFrame(detailed_results), summary_df


# 使用例
movie_num = "005"
test_images_dir=f"./movie_frame/movie_frame_{movie_num}"
detailed_results, summary_df = evaluate_multi_person_detailed(test_images_dir=test_images_dir,face_db=face_db,threshold=0.05,movie_num=movie_num,model_name='ArcFace')
summary_df.to_clipboard(index=False, header=False)
summary_df

=== 複数人テスト画像 詳細評価 ===

Image                正解              予測              検出    セット   ラベル   
---------------------------------------------------------------------------
frame_00000.jpg      kaki            Unknown         ✓     ✗     ✗     
frame_00001.jpg      kaki            suzuki          ✓     ✗     ✗     
frame_00002.jpg      kaki            umezawa         ✓     ✗     ✗     
frame_00003.jpg      kaki            suzuki          ✓     ✗     ✗     
frame_00004.jpg      kaki            suzuki          ✓     ✗     ✗     
frame_00005.jpg      kaki            yumiki          ✓     ✗     ✗     
frame_00006.jpg      kaki            Unknown         ✓     ✗     ✗     
frame_00007.jpg      kaki            Unknown         ✓     ✗     ✗     
frame_00008.jpg      kaki            Unknown         ✓     ✗     ✗     
frame_00009.jpg      kaki            suzuki          ✓     ✗     ✗     
frame_00010.jpg      kaki            hayashi         ✓     ✗     ✗     
frame_00011.jpg      kaki            

,detection_count,detection_rate,set_count,set_rate,label_count,label_rate
0,46,100.0,0,0.0,0,0.0


In [17]:
summary_df.to_clipboard(index=False, header=False)

In [13]:
def evaluate_multi_person_detailed_maxmatrix(test_images_dir, threshold, face_db_all, movie_num, model_name='Facenet'):
    """全画像DB(ラベル->np.array(n,d))を使って、各顔とフォルダ内の全画像を行列積で比較し、"""
    face_db_all = normalize_face_db_keys(face_db_all)

    anno_file = os.path.join(test_images_dir, f"annotations_{movie_num}.csv")
    if not os.path.exists(anno_file):
        print(f"⚠️ {anno_file} が見つかりません")
        return None, None

    anno_df = pd.read_csv(anno_file)

    detection_correct = 0
    set_correct = 0
    label_correct = 0
    detailed_results = []

    print("=== (max-matrix) 複数人テスト画像 詳細評価 ===\n")
    print(f"{'Image':<20} {'正解':<15} {'予測':<15} {'検出':<5} {'セット':<5} {'ラベル':<6}")
    print("-" * 75)

    for _, row in anno_df.iterrows():
        img_name = row['image_name']
        img_path = os.path.join(test_images_dir, img_name)
        if not os.path.exists(img_path): continue

        gt_persons = sorted([p.strip() for p in str(row['persons']).split('|') if p.strip()])
        try:
            objs = DeepFace.represent(img_path=img_path, model_name=model_name, detector_backend='retinaface', enforce_detection=False)
            from types import SimpleNamespace
            faces = []
            for o in objs:
                if isinstance(o, dict) and 'embedding' in o:
                    emb = np.asarray(o['embedding'], dtype=float)
                    norm = np.linalg.norm(emb)
                    normed = emb / norm if norm != 0 else emb
                    det = float(o.get('confidence', o.get('det_score', 1.0)))
                    faces.append(SimpleNamespace(det_score=det, normed_embedding=normed))
                else:
                    faces.append(o)

            # ここが差分: 全画像DBを使ったマトリクス方式でラベル割当
            pred_persons = assign_unique_person_labels_max_matrix(faces, face_db_all, threshold)

            detection_ok = (len(pred_persons) == len(gt_persons))
            set_ok = (set(pred_persons) == set(gt_persons))
            label_ok = (pred_persons == gt_persons)

            if detection_ok: detection_correct += 1
            if set_ok: set_correct += 1
            if label_ok: label_correct += 1

            det_mark = "✓" if detection_ok else "✗"
            set_mark = "✓" if set_ok else "✗"
            label_mark = "✓" if label_ok else "✗"

            gt_str = '|'.join(gt_persons)
            pred_str = '|'.join(pred_persons)
            print(f"{img_name:<20} {gt_str:<15} {pred_str:<15} {det_mark:<5} {set_mark:<5} {label_mark:<6}")

            detailed_results.append({'image': img_name, 'gt': gt_str, 'pred': pred_str, 'detection': detection_ok, 'set_match': set_ok, 'label_match': label_ok})
        except Exception as e:
            import traceback
            print(f"{img_name:<20} Error: {e}")
            traceback.print_exc()

    total = len(anno_df)
    print("\n" + "="*50)
    print(f"📊 集計結果 (計 {total} 画像)")
    print("="*50)
    print(f"顔検出精度 (人数一致):    {detection_correct}/{total} ({100*detection_correct/total:.1f}%)")
    print(f"セット照合精度 (順不同):    {set_correct}/{total} ({100*set_correct/total:.1f}%)")
    print(f"ラベル精度 (順序含む完全一致): {label_correct}/{total} ({100*label_correct/total:.1f}%)")

    summary_df = pd.DataFrame([{'detection_count': detection_correct, 'detection_rate': 100 * detection_correct / total if total else 0.0, 'set_count': set_correct, 'set_rate': 100 * set_correct / total if total else 0.0, 'label_count': label_correct, 'label_rate': 100 * label_correct / total if total else 0.0}])
    return pd.DataFrame(detailed_results), summary_df

# 使用例（全画像DB版）
movie_num = "007"
test_images_dir=f"./movie_frame/movie_frame_{movie_num}"
#face_db_all = get_insightface_all_embeddings_db('./', target_folders)
detailed_results_max, summary_max = evaluate_multi_person_detailed_maxmatrix(test_images_dir=test_images_dir, face_db_all=face_db_all, threshold=0.05, movie_num=movie_num, model_name='ArcFace')
summary_max.to_clipboard(index=False, header=False)
summary_max

=== (max-matrix) 複数人テスト画像 詳細評価 ===

Image                正解              予測              検出    セット   ラベル   
---------------------------------------------------------------------------
frame_00000.jpg      endou|ikeda|ioki endou|suzuki|umezawa ✓     ✗     ✗     
frame_00001.jpg      endou|ikeda|ioki ikeda|suzuki|umezawa ✓     ✗     ✗     
frame_00002.jpg      endou|ikeda|ioki ikeda|suzuki|umezawa ✓     ✗     ✗     
frame_00003.jpg      endou|ikeda|ioki suzuki|umezawa  ✗     ✗     ✗     
frame_00004.jpg      endou|ikeda|ioki ikeda|suzuki|umezawa ✓     ✗     ✗     
frame_00005.jpg      endou|ikeda|ioki Unknown|endou|suzuki ✓     ✗     ✗     
frame_00006.jpg      endou|ikeda|ioki Unknown|suzuki|umezawa ✓     ✗     ✗     
frame_00007.jpg      endou|ikeda|ioki endou|suzuki|umezawa ✓     ✗     ✗     
frame_00008.jpg      endou|ikeda|ioki Unknown|suzuki|umezawa ✓     ✗     ✗     
frame_00009.jpg      endou|ikeda|ioki endou|suzuki|umezawa ✓     ✗     ✗     
frame_00010.jpg      endou|ikeda|ioki

,detection_count,detection_rate,set_count,set_rate,label_count,label_rate
0,39,88.636364,0,0.0,0,0.0


In [11]:
summary_max.to_clipboard(index=False, header=False)

In [16]:
detailed_results.to_clipboard()